In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline  
import seaborn as sns
import sklearn  
import warnings
warnings.filterwarnings('ignore')   


In [ ]:
from sklearn.datasets import fetch_openml   
elec_data = fetch_openml(name='electricity', version=1)
elec_df=elec_data.frame



In [ ]:


print("DATA SHAPE:", elec_data.data.shape)      
print("FEATURE NAMES:", elec_data.feature_names) 
print("TARGET NAMES:", elec_data.target_names)

In [ ]:
from sklearn.model_selection import train_test_split
train_data, test_data   = train_test_split(elec_df, test_size=0.2, random_state=42)
print("TRAINING Samples:", len(train_data))
print("TESTING Samples:", len(test_data)) 


In [ ]:
train_data.head(10)

In [ ]:
train_data.info()

In [ ]:
train_data.describe
train_data.isnull().sum()

In [ ]:
corr=train_data.corr(numeric_only=True)
plt.figure(figsize=(12,8))  
sns.heatmap(corr, annot=True, cmap='coolwarm')  
plt.show()

In [ ]:
plt.figure(figsize=(12,8))  
sns.countplot(x='class', data=train_data)
plt.show()

In [ ]:
train_data.hist(bins=50, figsize=(20,15))
plt.show()  

In [ ]:
from sklearn.preprocessing import LabelEncoder  
X_train = train_data.drop('class', axis=1)  
y_train = train_data['class']   

label_enc= LabelEncoder()
y_train_prepared=label_enc.fit_transform(y_train)
print("Encoded Classes:", label_enc.classes_)

In [ ]:
from sklearn.metrics import accuracy_score

def accuracy(model,data,labels):
    y_pred=model.predict(data)
    acc= accuracy_score(labels,y_pred)
    return acc

In [ ]:
from sklearn.ensemble import VotingClassifier   ,RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

X_train_num=X_train.copy()
X_train_num['day']=X_train_num['day'].astype(int)
models=[('lr',LogisticRegression(max_iter=1000)), 
        ('rf',RandomForestClassifier(n_estimators=100, random_state=42)),
        ('svc',SVC(probability=True, random_state=42))]
voting_hard=VotingClassifier(models, voting='hard')
voting_soft=VotingClassifier(models, voting='soft') 
voting_hard.fit(X_train_num, y_train_prepared)
voting_soft.fit(X_train_num, y_train_prepared)
print("Hard Voting Classifier Accuracy:", accuracy(voting_hard, X_train_num, y_train_prepared)) 
print("Soft Voting Classifier Accuracy:", accuracy(voting_soft, X_train_num, y_train_prepared)) 

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier 
bag_classifier=BaggingClassifier(DecisionTreeClassifier(class_weight='balanced' ), max_samples=0.5, max_features=0.5,bootstrap=false )
bag_classifier.fit(X_train, y_train_prepared)   
print("Bagging Classifier Accuracy:", accuracy(bag_classifier, X_train, y_train_prepared))  

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier 
grad_boost_clf=GradientBoostingClassifier(n_estimators=100, learning_rate=0.8, max_depth=2, random_state=42)  
grad_boost_clf.fit(X_train, y_train_prepared)
print("Gradient Boosting Classifier Accuracy:", accuracy(grad_boost_clf, X_train,y_train_prepared))  

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
ada_boost_clf = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=3,class_weight='balanced' ), n_estimators=300, learning_rate=0.5  )

In [ ]:
import xgboost as xgb   
xgb_clf = xgb.XGBClassifier()
xgb_clf.fit(X_train, y_train_prepared)
print("XGBoost Classifier Accuracy:", accuracy(xgb_clf, X_train, y_train_prepared)) 

In [ ]:
from sklearn.ensemble import StackingClassifier,RandomForestClassifier
from sklearn.tree   import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('dt', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('lr', LogisticRegression(max_iter=1000))   
]
stacking_clf = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression(max_iter=1000  ),cv=5) 
stacking_clf.fit(X_train, y_train_prepared)
print("Stacking Classifier Accuracy:", accuracy(stacking_clf, X_train, y_train_prepared))

In [ ]:
X_test = test_data.drop('class', axis=1)    
y_test = test_data['class'] 
y_test_prepared=label_enc.transform(y_test) 
print(accuracy(grad_boost_clf, X_test, y_test_prepared  ))
print(accuracy(bag_classifier, X_test, y_test_prepared  ))

In [ ]:
X_test_num = X_test.copy()
X_test_num['day'] = X_test_num['day'].astype(int)

print(accuracy(voting_hard, X_test_num, y_test_prepared))
print(accuracy(voting_soft, X_test_num, y_test_prepared))
print(accuracy(xgb_clf, X_test_num, y_test_prepared))
print(accuracy(stack_clf, X_test_num, y_test_prepared))
print(accuracy(adaboost_clf, X_test, y_test_prepared))

